# Phase 5A — Crime Hotspot Analysis (KDE)

**Method:** Kernel Density Estimation (KDE) applied to 1,002,420 crime coordinates over a 250×250 grid covering the LA bounding box.

**Bandwidth:** 0.018 degrees (~2 km), chosen to balance local detail vs. noise.

**Outputs:**
- Overall LA hotspot heatmap 2020-2024
- Year-by-year evolution (normalized to common scale)
- Hotspot by top-6 crime categories
- Violent vs Property spatial comparison
- Discretised risk-tier map (Low / Medium / High) + grid export

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
import subprocess, sys

ROOT   = Path('..').resolve()
PROC   = ROOT / 'data' / 'processed'
EXT    = ROOT / 'data' / 'external'
REPDIR = ROOT / 'outputs' / 'reports'
FIGDIR = ROOT / 'outputs' / 'figures'

print('Setup complete.')

## 0. Run the hotspot pipeline

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'src' / 'ml_hotspot.py')],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## 1. Overall Crime Hotspot Map — 2020-2024

In [ ]:
display(Image(filename=str(FIGDIR / 'p5a_01_hotspot_overall.png'), width=800))

**Findings:**
- **Downtown / Central Division** (−118.25, 34.05) shows the highest density peak — consistent with Skid Row concentration
- **Hollywood / Wilshire corridor** forms a secondary hotspot band running east-west
- **77th Street / Southeast / Newton** divisions create a high-density cluster in South LA
- **Topanga / Foothill** (north) and coastal areas show relatively low density — residential, lower-crime zones

## 2. Year-by-Year Evolution

In [ ]:
display(Image(filename=str(FIGDIR / 'p5a_02_hotspot_by_year.png'), width=1000))

**Temporal dynamics:**
- **2020:** Reduced overall density (COVID lockdowns reduced mobility and outdoor crime)
- **2021–2022:** Rebound — density approaches pre-pandemic levels
- **2023–2024:** Hotspot geography stabilizes; Central Division remains consistently the apex

## 3. Hotspot by Crime Category

In [ ]:
display(Image(filename=str(FIGDIR / 'p5a_03_hotspot_by_category.png'), width=1000))

**Category-specific geography:**
- **Vehicle Theft:** Diffuse across all of LA — no single dominant cluster; reflects citywide street parking exposure
- **Theft / Burglary:** Concentrated in commercial corridors (Hollywood, Westside, Downtown)
- **Assault:** High-density in Central, 77th Street, Rampart — poverty-concentrated areas
- **Robbery:** Central + South LA cluster; also elevated around transit hubs

## 4. Violent vs Property Crime Distribution

In [ ]:
display(Image(filename=str(FIGDIR / 'p5a_04_hotspot_violent.png'), width=1000))

**Key observation:** Violent and property crime hotspots diverge significantly:
- Violent crime: tightly clustered in South LA, Central, and Rampart divisions
- Property crime: more diffuse, extends into Westside commercial zones (Santa Monica, Wilshire)

## 5. Risk Tier Map

In [ ]:
display(Image(filename=str(FIGDIR / 'p5a_05_risk_grid.png'), width=800))

## 6. Risk Grid Statistics

In [ ]:
grid = pd.read_parquet(REPDIR / 'hotspot_grid.parquet')
print(f'Grid cells (non-zero density): {len(grid):,}')
print(f'\nRisk tier distribution:')
tier_labels = {1: 'Very Low', 2: 'Low', 3: 'Medium', 4: 'High'}
for tier, cnt in grid['risk_tier'].value_counts().sort_index().items():
    pct = cnt / len(grid) * 100
    print(f'  Tier {tier} ({tier_labels.get(tier,"?")}): {cnt:,} cells ({pct:.1f}%)')
    
high_risk = grid[grid['risk_tier'] == 4]
print(f'\nHigh-risk cells: {len(high_risk):,}')
print(f'  Lat range: {high_risk["lat"].min():.4f} to {high_risk["lat"].max():.4f}')
print(f'  Lon range: {high_risk["lon"].min():.4f} to {high_risk["lon"].max():.4f}')
print(f'  Central point: ({high_risk["lat"].mean():.4f}, {high_risk["lon"].mean():.4f})')

---
## Key Findings & ML Implications

| Insight | Implication for Prediction Models |
|---------|-----------------------------------|
| Crime density is spatially autocorrelated | `LAT`, `LON`, `AREA` are strong features for classifiers |
| High-risk zone = top 10% of density (~25 cells) | KDE risk score can be a feature in the forecast model |
| Category hotspots diverge significantly | Separate spatial models per category outperform a global model |
| 2020 COVID dip is spatial, not random | Year + COVID indicator variable essential for time series |
| Violent crime clusters tighter than property | Different bandwidth (tighter) optimal for violent-only KDE |

**Risk grid exported:** `outputs/reports/hotspot_grid.parquet` — 62,500 cells with lat/lon and risk tier, ready for Power BI or Vercel dashboard integration.